## Load all datasets as Data Frames

In [1]:
import pandas as pd
import re
import numpy as np

In [2]:
splits = {'explicit_train': 'AbuseEval/explicit_train.jsonl', 'explicit_test': 'AbuseEval/explicit_test.jsonl', 'implicit_train': 'AbuseEval/implicit_train.jsonl', 'implicit_test': 'AbuseEval/implicit_test.jsonl'}

df_AbuseEval_explicit_train = pd.read_json("hf://datasets/Shuwan/cadet-datasets/" + splits["explicit_train"], lines=True)
df_AbuseEval_explicit_test = pd.read_json("hf://datasets/Shuwan/cadet-datasets/" + splits["explicit_test"], lines=True)
df_AbuseEval_implicit_train = pd.read_json("hf://datasets/Shuwan/cadet-datasets/" + splits["implicit_train"], lines=True)
df_AbuseEval_implicit_test = pd.read_json("hf://datasets/Shuwan/cadet-datasets/" + splits["implicit_test"], lines=True)

In [3]:
splits = {'explicit_train': 'DynaHate/explicit_train.jsonl', 'explicit_test': 'DynaHate/explicit_test.jsonl', 'implicit_train': 'DynaHate/implicit_train.jsonl', 'implicit_test': 'DynaHate/implicit_test.jsonl'}

df_DynaHate_explicit_train = pd.read_json("hf://datasets/Shuwan/cadet-datasets/" + splits["explicit_train"], lines=True)
df_DynaHate_explicit_test = pd.read_json("hf://datasets/Shuwan/cadet-datasets/" + splits["explicit_test"], lines=True)
df_DynaHate_implicit_train = pd.read_json("hf://datasets/Shuwan/cadet-datasets/" + splits["implicit_train"], lines=True)
df_DynaHate_implicit_test = pd.read_json("hf://datasets/Shuwan/cadet-datasets/" + splits["implicit_test"], lines=True)

In [4]:
splits = {'explicit_train': 'Implicit-Hate-Corpus/explicit_train.jsonl', 'explicit_test': 'Implicit-Hate-Corpus/explicit_test.jsonl', 'implicit_train': 'Implicit-Hate-Corpus/implicit_train.jsonl', 'implicit_test': 'Implicit-Hate-Corpus/implicit_test.jsonl'}

df_Implicit_Hate_Corpus_explicit_train = pd.read_json("hf://datasets/Shuwan/cadet-datasets/" + splits["explicit_train"], lines=True)
df_Implicit_Hate_Corpus_explicit_test = pd.read_json("hf://datasets/Shuwan/cadet-datasets/" + splits["explicit_test"], lines=True)
df_Implicit_Hate_Corpus_implicit_train = pd.read_json("hf://datasets/Shuwan/cadet-datasets/" + splits["implicit_train"], lines=True)
df_Implicit_Hate_Corpus_implicit_test = pd.read_json("hf://datasets/Shuwan/cadet-datasets/" + splits["implicit_test"], lines=True)

In [5]:
splits = {'explicit_train': 'IsHate/explicit_train.jsonl', 'explicit_test': 'IsHate/explicit_test.jsonl', 'implicit_train': 'IsHate/implicit_train.jsonl', 'implicit_test': 'IsHate/implicit_test.jsonl'}
df_IsHate_explicit_train = pd.read_json("hf://datasets/Shuwan/cadet-datasets/" + splits["explicit_train"], lines=True)
df_IsHate_explicit_test = pd.read_json("hf://datasets/Shuwan/cadet-datasets/" + splits["explicit_test"], lines=True)
df_IsHate_implicit_train = pd.read_json("hf://datasets/Shuwan/cadet-datasets/" + splits["implicit_train"], lines=True)
df_IsHate_implicit_test = pd.read_json("hf://datasets/Shuwan/cadet-datasets/" + splits["implicit_test"], lines=True)

## Find all Target values

In [6]:
all_dfs = [df_AbuseEval_explicit_train, df_AbuseEval_explicit_test, df_AbuseEval_implicit_train, df_AbuseEval_implicit_test,
           df_DynaHate_explicit_train, df_DynaHate_explicit_test, df_DynaHate_implicit_train, df_DynaHate_implicit_test,
           df_Implicit_Hate_Corpus_explicit_train, df_Implicit_Hate_Corpus_explicit_test, df_Implicit_Hate_Corpus_implicit_train, df_Implicit_Hate_Corpus_implicit_test,
           df_IsHate_explicit_train, df_IsHate_explicit_test, df_IsHate_implicit_train, df_IsHate_implicit_test]  

all_dfs_concat = pd.concat(all_dfs, ignore_index=True)
all_dfs_concat['target'].unique()

array(['Abnormality', 'Gender', 'Race', ..., 'Hunting knives', 'Life',
       'Motorrad'], shape=(1326,), dtype=object)

In [7]:
# Show all unique values without truncation
import numpy as np
np.set_printoptions(threshold=np.inf)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
print(all_dfs_concat['target'].unique())

['Abnormality' 'Gender' 'Race' 'Immigration Status' 'Abnormal Activity'
 'Religion' 'Relationship' 'Mental Health'
 'Violence and Threat to Government' 'Violence'
 'Sexuality and Sexual Preferences' 'none' 'Anger' 'Politics' 'Aggression'
 'Fascism' 'Sexual Preferences' 'Alcoholism' 'Emotion' 'Sports' 'Abnormal'
 'Abnormal Behavior' 'Abelness/Disability' 'Gun Control' 'Abnormal System'
 'Abnormal_Emotion' 'Social unrest' 'Sex and Sexual Preferences'
 'Sexual and Sexual Preferences' 'Narcissism' 'Security' 'Crime'
 'Mobility' 'Slowness' 'Corruption' 'Gun control' 'Abuse' 'Furniture'
 'Stalking' 'Hungry' 'Sarcasm' 'Electrical' 'Health' 'Terrorism' 'Unknown'
 'Animal' 'Sociopath' 'Hair' 'Abience/Disability' 'Irrelevant' 'Rape'
 'Threat' 'Social Issues' 'Food & Dining' 'Abhorrent' 'Abortion'
 'Abnormal Language' 'Hate Speech' 'Comedy' 'Sexual Misconduct' 'Dementia'
 'Entertainment' 'Class' 'Domestic Abuse' 'Bullying' 'Roads'
 'Substance Abuse' 'Music' 'Abduction' 'Beauty' 'Europe' 'Philosop

## Normalize all target values to obtain uniform Data Frames

In [8]:
import re

def normalize_target(x):
    if pd.isna(x):
        return "none"

    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)

    if x in ["none", "unknown", "irrelevant", "miscellaneous", "all", "n"]:
        return "none"

    # Race / ethnicity / racism / specific ethnic references
    if any(k in x for k in [
        "race", "racism", "skin tone", "skin color", "colour", "color",
        "native american", "aboriginal", "mexican", "korean", "irish",
        "japanese", "sami", "ghetto", "nigger"
    ]):
        return "race"

    # Religion
    if any(k in x for k in [
        "religion", "islam", "judaism", "christian", "jihad",
        "islamophobia", "holocaust", "anti-semitism", "satanism"
    ]):
        return "religion"

    # Gender / women / feminism
    if any(k in x for k in [
        "gender", "male", "female", "women", "woman", "feminism",
        "motherhood", "pregnancy", "breastfeeding"
    ]):
        return "gender"

    # Sexual orientation / LGBTQ+ / trans
    if any(k in x for k in [
        "sexual", "homosexual", "lgbt", "queer", "trans",
        "bisexual", "lesbian", "homophobia", "transphobia",
        "gay", "heterosexual"
    ]):
        return "sexual_orientation"

    # Immigration / refugees / asylum
    if any(k in x for k in [
        "immigration", "migration", "refugee", "asylum",
        "visa", "border"
    ]):
        return "immigration_status"

    # Nationality / nation / nationalism
    if any(k in x for k in [
        "nationality", "nationals", "citizenship", "nationalism",
        "country", "england", "romania", "poland", "europe"
    ]):
        return "nationality"

    # Disability / ableism / mental disability
    if any(k in x for k in [
        "disability", "ableness", "abless", "abelness", "abbling",
        "autism", "autistic", "blindness", "paralysis", "spastic",
        "dyslexia", "asperger"
    ]):
        return "disability"

    # Social class / poverty / welfare / homelessness
    if any(k in x for k in [
        "class", "poverty", "homeless", "welfare", "income",
        "wealth", "unemployment", "rent avoiders"
    ]):
        return "class"

    # Politics / ideology / government
    if any(k in x for k in [
        "politic", "government", "election", "democracy", "socialism",
        "communism", "marxism", "antifa", "fascism", "national socialism",
        "populism", "brexit"
    ]):
        return "politics"

    # Violence / crime / threats / war / terrorism / weapons
    if any(k in x for k in [
        "violence", "threat", "terror", "war", "crime", "criminal",
        "rape", "murder", "homicide", "abduction", "assault",
        "gun", "weapon", "knife", "gang", "attack", "riot",
        "nuclear", "genocide", "combat", "military"
    ]):
        return "violence"

    # Health / medical / mental health / substance
    if any(k in x for k in [
        "health", "mental", "illness", "disease", "medical",
        "cancer", "diabetes", "covid", "coronavirus", "virus",
        "suicide", "self-harm", "addiction", "substance", "alcohol"
    ]):
        return "health"

    return "other"

In [9]:
for df in all_dfs:
    df["target"] = df["target"].apply(normalize_target)

## Creating the Ultimate Dataset

In [10]:
df_Ultimate_explicit_train = pd.concat([df_AbuseEval_explicit_train, df_DynaHate_explicit_train, df_Implicit_Hate_Corpus_explicit_train, df_IsHate_explicit_train], ignore_index=True)
df_Ultimate_explicit_test = pd.concat([df_AbuseEval_explicit_test, df_DynaHate_explicit_test, df_Implicit_Hate_Corpus_explicit_test, df_IsHate_explicit_test], ignore_index=True)
df_Ultimate_implicit_train = pd.concat([df_AbuseEval_implicit_train, df_DynaHate_implicit_train, df_Implicit_Hate_Corpus_implicit_train, df_IsHate_implicit_train], ignore_index=True)
df_Ultimate_implicit_test = pd.concat([df_AbuseEval_implicit_test, df_DynaHate_implicit_test, df_Implicit_Hate_Corpus_implicit_test, df_IsHate_implicit_test], ignore_index=True)

In [11]:
#df_Ultimate_explicit_train.to_csv("ultimate_explicit_train.csv", index=False)
#df_Ultimate_explicit_test.to_csv("ultimate_explicit_test.csv", index=False)
#df_Ultimate_implicit_train.to_csv("ultimate_implicit_train.csv", index=False)
#df_Ultimate_implicit_test.to_csv("ultimate_implicit_test.csv", index=False)

In [12]:
df_Ultimate_explicit = pd.concat([df_Ultimate_explicit_train, df_Ultimate_explicit_test], ignore_index=True)
df_Ultimate_implicit = pd.concat([df_Ultimate_implicit_train, df_Ultimate_implicit_test], ignore_index=True)
df_Ultimate_combined = pd.concat([df_Ultimate_explicit, df_Ultimate_implicit], ignore_index=True)

In [13]:
#df_Ultimate_explicit.to_csv("ultimate_explicit.csv", index=False)
#df_Ultimate_implicit.to_csv("ultimate_implicit.csv", index=False)
#df_Ultimate_combined.to_csv("ultimate_combined.csv", index=False)

In [14]:
df_Ultimate_implicit["hate_label"].value_counts()

hate_label
1    56348
0    52555
Name: count, dtype: int64

## Creating the Gab Test Dataset

In [15]:
df_GAB_raw = pd.read_csv("GabHateCorpus_annotations.tsv", sep="\t")

In [16]:
df_GAB_raw.head(10)

,ID,Annotator,Text,Hate,HD,CV,VO,REL,RAE,SXO,GEN,IDL,NAT,POL,MPH,EX,IM
0,27044,4,Ah the PSYOPS antifa crew is back. That’s how I know we are winning.,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,27044,15,Ah the PSYOPS antifa crew is back. That’s how I know we are winning.,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,27044,10,Ah the PSYOPS antifa crew is back. That’s how I know we are winning.,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,27044,8,Ah the PSYOPS antifa crew is back. That’s how I know we are winning.,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,27045,4,Get the new Android app update released today here: https://gab.ai/about/android,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,27045,10,Get the new Android app update released today here: https://gab.ai/about/android,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,27045,8,Get the new Android app update released today here: https://gab.ai/about/android,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,27046,4,Study: Google Pushes Liberal News in Top 5 Search Suggestions https://www.newsbusters.org/blogs/techwatch/corinn...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,27046,15,Study: Google Pushes Liberal News in Top 5 Search Suggestions https://www.newsbusters.org/blogs/techwatch/corinn...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,27046,10,Study: Google Pushes Liberal News in Top 5 Search Suggestions https://www.newsbusters.org/blogs/techwatch/corinn...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
import random

rng = random.Random(42)

def gab_style(row):
    if row["EX"] >= 0.5:
        return 1  # explicit
    if row["IM"] >= 0.5:
        return 0  # implicit
    return np.nan

def gab_targets_all(row):
    target_cols = {
        "RAE": "race",
        "NAT": "nationality",
        "GEN": "gender",
        "REL": "religion",
        "SXO": "sexual_orientation",
        "IDL": "politics",
        "POL": "politics",
        "MPH": "health",
    }

    active = [
        canonical
        for col, canonical in target_cols.items()
        if row.get(col, 0) == 1
    ]

    if row.get("CV", 0) == 1:
        active.append("violence")

    if not active:
        if row["Hate"] == 1:
            return ["other"]
        else:
            return ["none"]

    return list(set(active))  # remove duplicates

In [18]:
df_GAB = pd.DataFrame()
df_GAB["text"] = df_GAB_raw["Text"]
df_GAB["hate_label"] = df_GAB_raw["Hate"].astype(int)
df_GAB["style"] = df_GAB_raw.apply(gab_style, axis=1)
df_GAB["target_list"] = df_GAB_raw.apply(gab_targets_all, axis=1)
df_GAB["target_conf"] = 1.0

# Only keep examples with known style
df_GAB_all = df_GAB.copy() # dataset used for text and hate label only
df_GAB = df_GAB.dropna(subset=["style"])
df_GAB["style"] = df_GAB["style"].astype(int)

# Clean text column
df_GAB = df_GAB.dropna(subset=["text"])
df_GAB_all["text"] = df_GAB_all["text"].astype(str)
df_GAB["text"] = df_GAB["text"].astype(str)

In [19]:
df_GAB.head(10)

,text,hate_label,style,target_list,target_conf
14,"#Breaking: Claiming bias against conservatives, @FreedomWatchUSA has filed a class action lawsuit against Google, Facebook, Instagram, Twitter, and Apple.",1,0,[politics],1.0
16,"“These [companies] are run by sociopaths,” he said. “These people are complete narcissists. These people ought to be controlled, they ought to be regulated.” At one point during the phone call, Bannon said, “These people are evil.' https://www.vanityfair.com/news/2018/08/steve-bann...",1,1,"[nationality, politics]",1.0
54,"Report: Google A.I. Is ‘Biased’ Against Gay People, Jews http://www.breitbart.com/tech/2017/10/26/report-go...",1,1,"[sexual_orientation, religion]",1.0
67,"Except, perhaps, to #SmugLiberal awards ceremonies...or maybe out back to shoot 'em. #SmugHollywoodTypes #Hollywood #JimmyKimmel #Trump #Hannity #Homophobic #Twitter #Twatter #OutBackToShootEm",1,1,"[violence, politics]",1.0
118,'Who’s paying your salary? The Italians who live here? Or the people getting off boats tomorrow morning?' —Matteo Salvini scolds Khalid Chaouki in a heated debate. https://youtu.be/F1N5QLmzCQA #Mastrangelo #Salvini #Italy #gimmegrants #,1,0,[nationality],1.0
132,#GabBrasil an enterprising capitalist might do well setting up an online course/forum teaching Portuguese to Deplorable US Patriots. And/or vice versa... Just saying. 😎,1,1,[nationality],1.0
194,Faggot. https://archive.fo/Kb0YP#selection-457.0-1061.1,1,1,[sexual_orientation],1.0
197,Faggot. https://archive.fo/Kb0YP#selection-457.0-1061.1,1,1,[sexual_orientation],1.0
243,"An inflated example of gesture politics is all it is, and a pathetic reminder of how many on the left have descended to the same level as the target of their protest. https://medium.com/@sd_saurav/the-trump-blimp-woul...",1,1,[politics],1.0
245,My interview with the one and only @GeertWilders on #MuhammadCartoonContest sent your entry in yet?? https://medium.com/@sd_saurav/why-geert-wilders-is...,1,0,[religion],1.0


In [20]:
df_GAB.head(10)

,text,hate_label,style,target_list,target_conf
14,"#Breaking: Claiming bias against conservatives, @FreedomWatchUSA has filed a class action lawsuit against Google, Facebook, Instagram, Twitter, and Apple.",1,0,[politics],1.0
16,"“These [companies] are run by sociopaths,” he said. “These people are complete narcissists. These people ought to be controlled, they ought to be regulated.” At one point during the phone call, Bannon said, “These people are evil.' https://www.vanityfair.com/news/2018/08/steve-bann...",1,1,"[nationality, politics]",1.0
54,"Report: Google A.I. Is ‘Biased’ Against Gay People, Jews http://www.breitbart.com/tech/2017/10/26/report-go...",1,1,"[sexual_orientation, religion]",1.0
67,"Except, perhaps, to #SmugLiberal awards ceremonies...or maybe out back to shoot 'em. #SmugHollywoodTypes #Hollywood #JimmyKimmel #Trump #Hannity #Homophobic #Twitter #Twatter #OutBackToShootEm",1,1,"[violence, politics]",1.0
118,'Who’s paying your salary? The Italians who live here? Or the people getting off boats tomorrow morning?' —Matteo Salvini scolds Khalid Chaouki in a heated debate. https://youtu.be/F1N5QLmzCQA #Mastrangelo #Salvini #Italy #gimmegrants #,1,0,[nationality],1.0
132,#GabBrasil an enterprising capitalist might do well setting up an online course/forum teaching Portuguese to Deplorable US Patriots. And/or vice versa... Just saying. 😎,1,1,[nationality],1.0
194,Faggot. https://archive.fo/Kb0YP#selection-457.0-1061.1,1,1,[sexual_orientation],1.0
197,Faggot. https://archive.fo/Kb0YP#selection-457.0-1061.1,1,1,[sexual_orientation],1.0
243,"An inflated example of gesture politics is all it is, and a pathetic reminder of how many on the left have descended to the same level as the target of their protest. https://medium.com/@sd_saurav/the-trump-blimp-woul...",1,1,[politics],1.0
245,My interview with the one and only @GeertWilders on #MuhammadCartoonContest sent your entry in yet?? https://medium.com/@sd_saurav/why-geert-wilders-is...,1,0,[religion],1.0


In [21]:
from collections import Counter

def majority_label(series):
    values = series.dropna().tolist()
    if len(values) == 0:
        return np.nan
    return Counter(values).most_common(1)[0][0]

def majority_target(series):
    all_targets = []

    for targets in series:
        all_targets.extend(targets)

    if len(all_targets) == 0:
        return "none"

    counts = Counter(all_targets)
    max_count = max(counts.values())

    top_targets = [k for k, v in counts.items() if v == max_count]

    # if multiple top targets, return random choice among them (tie-break)
    return rng.choice(top_targets)

df_GAB_agg = (
    df_GAB
    .groupby("text", as_index=False)
    .agg(
        avg=("hate_label", "mean"),
        hate_label=("hate_label", lambda x: int(x.mean() >= 0.5)),
        style=("style", majority_label),
        target=("target_list", majority_target),
        target_conf=("target_conf", "mean"),
    )
)

df_GAB_agg["style"] = df_GAB_agg["style"].astype(int)
df_GAB_agg["target_conf"] = 1.0

In [22]:
df_GAB_final = df_GAB_agg[
    ["text", "hate_label", "style", "target", "target_conf"]
].copy()

In [23]:
#df_GAB_final.to_csv("gab_combined.csv", index=False)

In [24]:
df_GAB_explicit = df_GAB_final[df_GAB_final["style"] == 1].copy()
df_GAB_implicit = df_GAB_final[df_GAB_final["style"] == 0].copy()

In [25]:
#df_GAB_explicit.to_csv("gab_explicit.csv", index=False)
#df_GAB_implicit.to_csv("gab_implicit.csv", index=False)

In [26]:
df_GAB_all.drop(columns=["style", "target_list", "target_conf"], inplace=True)
df_GAB_all_agg = df_GAB_all.groupby("text", as_index=False).agg(
    hate_label=("hate_label", lambda x: int(x.mean() >= 0.5))
)

In [27]:
#df_GAB_all_agg.to_csv("gab_only_hate_label.csv", index=False)

## Creating the SBIC test dataset

In [28]:
df_SBIC_raw = pd.read_csv("SBIC.v2.trn.csv")

In [29]:
np.set_printoptions(threshold=np.inf)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
df_SBIC_raw['targetMinority'].unique()

array([nan, 'black folks', 'women', 'black women', 'conservatives',
       'republicans', 'ugly folks', 'gay men', 'poor folks',
       'lesbian women', 'overweight/fat folks', 'white folks',
       'black folks, white folks', 'AAP',
       'folks with mental illness/disorder, mentally disabled folks',
       'mentally disabled folks', 'folks with mental illness/disorder',
       'latino/latina folks', 'lesbian women, gay men', 'muslim folks',
       'jewish folks, muslim folks', 'physically disabled folks',
       'women, gay men', 'black folks, white women', 'white females',
       'men',
       'gay men, trans women, trans men, bisexual women, bisexual men',
       'Islam', 'assault victims', 'physically disabled folks, fat folks',
       'Southerners', 'asian folks', 'rednecks', 'jewish folks',
       'rape victims',
       'lesbian women, gay men, trans women, trans men, bisexual women, bisexual men',
       'native american/first nation folks', 'immigrants', 'White people',
     

In [30]:
def normalize_target_SBIC(x):
    if pd.isna(x):
        return ["none"]

    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)

    if x in ["", "nan", "none", "unknown", "not specified", "everyone", "all", "all minorities"]:
        return ["none"]

    # Plusieurs targets séparées par virgules
    parts = [p.strip() for p in re.split(r",|/|;", x)]
    mapped = [normalize_single_target(p) for p in parts if p.strip()]

    # Enlever none si d'autres targets existent
    mapped = [m for m in mapped if m != "none"]

    if len(mapped) == 0:
        return ["none"]

    # Si plusieurs catégories, garder la première après mapping
    # ou remplacer par random/majority selon ton pipeline
    return mapped


def normalize_single_target(x):
    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)

    if x in ["", "nan", "none", "unknown", "not specified"]:
        return "none"

    # Race / ethnicity
    if any(k in x for k in [
        "black", "white", "asian", "latino", "latina", "hispanic",
        "native american", "first nation", "indigenous", "aboriginal",
        "poc", "people of color", "colored", "non-white", "non white",
        "nonwhite", "minority", "minorities", "mixed race", "biracial",
        "caucasian", "mexican", "arab", "middle eastern", "indian",
        "pakistani", "chinese", "japanese", "korean", "thai", "vietnamese",
        "african", "ethiopian", "somali", "nigerian", "jamaican",
        "gypsy", "romani", "roma", "slavic", "skin"
    ]):
        return "race"

    # Religion
    if any(k in x for k in [
        "muslim", "islam", "islamic", "jew", "jewish", "hebrew",
        "christian", "catholic", "mormon", "jehovah", "buddhist",
        "hindu", "atheist", "religion", "religious", "holocaust",
        "anti-semitism", "antisemitism"
    ]):
        return "religion"

    # Gender
    if any(k in x for k in [
        "women", "woman", "men", "man", "male", "female", "girls",
        "boys", "mothers", "fathers", "pregnant", "feminist",
        "feminists", "gender neutral", "non-binary", "nonbinary",
        "third gender", "genderqueer", "cis", "breast", "penis"
    ]):
        return "gender"

    # Sexual orientation / LGBTQ+
    if any(k in x for k in [
        "gay", "lesbian", "bisexual", "trans", "lgbt", "lgbtq",
        "queer", "homosexual", "homosexuals", "asexual",
        "sexual orientation", "sexuality", "heterosexual",
        "incel", "incels", "virgin", "sex workers", "prostitutes",
        "porn stars", "hookers"
    ]):
        return "sexual_orientation"

    # Immigration / refugees
    if any(k in x for k in [
        "immigrant", "immigrants", "illegal immigrants", "refugee",
        "refugees", "dreamers", "foreigners", "asylum"
    ]):
        return "immigration_status"

    # Nationality / regional origin
    if any(k in x for k in [
        "american", "americans", "canada", "canadian", "england",
        "british", "irish", "scottish", "welsh", "french", "german",
        "italian", "polish", "greek", "turkish", "syrian", "syria",
        "russian", "russia", "ukrainian", "ukrainians", "iranian",
        "afghan", "afghanistan", "palestinian", "palestine", "israel",
        "israeli", "saudi", "iraqi", "kurds", "romania", "romanians",
        "albanian", "albanians", "brazilian", "brazilians",
        "venezuelan", "venezuelans", "malaysian", "cambodian",
        "puerto rican", "sudanese", "kenyan", "kenyans", "texans",
        "southerners", "southern", "alabama", "rural americans",
        "nationality", "country", "european", "europeans", "asia",
        "africa", "middle east", "north korea", "south korea",
        "india", "pakistan", "china", "japan", "mexico"
    ]):
        return "nationality"

    # Disability / physical or mental disability
    if any(k in x for k in [
        "disabled", "disability", "physically disabled", "mentally disabled",
        "mental illness", "mental disorder", "ocd", "autistic", "autism",
        "down syndrome", "dwarfism", "midget", "little people",
        "blind", "deaf", "speech impediment", "paralysis", "handicapped",
        "alzheimer", "parkinson", "leprosy"
    ]):
        return "disability"

    # Class / social status
    if any(k in x for k in [
        "poor", "poverty", "homeless", "rich", "well off", "working class",
        "class", "welfare", "starvation", "hunger"
    ]):
        return "class"

    # Politics / ideology
    if any(k in x for k in [
        "conservative", "conservatives", "republican", "republicans",
        "liberal", "liberals", "democrat", "democrats", "leftist",
        "right-wing", "alt right", "alt-right", "antifa", "anti-fa",
        "fascist", "fascists", "communist", "communists", "socialist",
        "socialists", "trump", "blm", "blacklivesmatter", "sjw",
        "social justice warrior", "politician", "politicians",
        "political", "politics", "ideology", "gun control advocates",
        "gun enthusiasts", "pro-life advocates", "climate deniers"
    ]):
        return "politics"

    # Violence / victims / crime / war
    if any(k in x for k in [
        "victim", "victims", "assault", "rape", "murder", "homicide",
        "molestation", "abuse", "domestic violence", "violence",
        "kidnap", "trafficking", "terrorism", "war", "genocide",
        "slavery", "slaves", "shooting", "bombing", "attack",
        "crime", "violent", "death", "dead", "suicide victims",
        "holocaust victims", "pearl harbor", "911", "chernobyl",
        "nuclear bombing", "disaster victims", "fire victims",
        "accident victims", "car crash", "plane crash", "waterboarding"
    ]):
        return "violence"

    # Health
    if any(k in x for k in [
        "health", "cancer", "patients", "hiv", "aids", "disease",
        "illness", "sick", "terminally ill", "anorexic", "anorexia",
        "bulimia", "suicidal", "self-harm", "addiction", "drug",
        "coma", "miscarriage", "sids", "ebola", "leprosy"
    ]):
        return "health"

    return "other"

In [31]:
df_SBIC = pd.DataFrame()
df_SBIC["text"] = df_SBIC_raw["post"]
df_SBIC["hate_label"] = (df_SBIC_raw["offensiveYN"] >= 0.5).astype(int)
df_SBIC["style"] = df_SBIC_raw['targetStereotype'].isna().apply(lambda x: 1 if x else 0)  # implicit if stereotype is present, explicit if not
df_SBIC["target_list"] = df_SBIC_raw['targetMinority'].apply(normalize_target_SBIC)
df_SBIC["target_conf"] = 1.0

# Clean text column
df_SBIC = df_SBIC.dropna(subset=["text"])
df_SBIC["text"] = df_SBIC["text"].astype(str)

In [32]:
from collections import Counter
import random

rng = random.Random(42)

def majority_label(series):
    values = series.dropna().tolist()
    if len(values) == 0:
        return np.nan
    return Counter(values).most_common(1)[0][0]

def majority_target(series):
    all_targets = []

    for targets in series:
        all_targets.extend(targets)

    if len(all_targets) == 0:
        return "none"

    counts = Counter(all_targets)
    max_count = max(counts.values())

    top_targets = [k for k, v in counts.items() if v == max_count]

    # if multiple top targets, return random choice among them (tie-break)
    return rng.choice(top_targets)

df_SBIC_agg = (
    df_SBIC
    .groupby("text", as_index=False)
    .agg(
        avg=("hate_label", "mean"),
        hate_label=("hate_label", lambda x: int(x.mean() >= 0.5)),
        style=("style", majority_label),
        target=("target_list", majority_target),
        target_conf=("target_conf", "mean"),
    )
)

df_SBIC_agg["style"] = df_SBIC_agg["style"].astype(int)
df_SBIC_agg["target_conf"] = 1.0

In [33]:
df_SBIC_final = df_SBIC_agg[
    ["text", "hate_label", "style", "target", "target_conf"]
].copy()

In [34]:
df_SBIC_implicit = df_SBIC_final[df_SBIC_final["style"] == 0].copy()
df_SBIC_explicit = df_SBIC_final[df_SBIC_final["style"] == 1].copy()

In [35]:
df_SBIC_final.drop(columns=["target", "target_conf"], inplace=True)
#df_SBIC_final.to_csv("sbic_combined.csv", index=False)

## HateCOT Dataset

We gave the following prompt to GPT 5.5:

"You are an expert annotator able to distinguish explicit and implicit text. I have the following HateCOT dataset. I want you to go through every text and explanation to determine if the text is explicit (direct, clear, and literal, leaving no doubt about the speaker's intent) or implicit (indirect, implied, or suggested, requiring to infer meaning from context or tone). Add 2 columns to the dataset: one style column containing 0 if the text is implicit, and 1 if it is explicit. The second column is style explanation where you briefly explain why it is implicit / explicit"

And itered this prompt 5 times, obtaining 5 datasets. We averaged the results to obtain a more stable style across texts and explanations.

In [70]:
df_hatecot_raw = pd.read_csv("hatecot_final_D3.csv")

In [71]:
df_hatecot_style_1 = pd.read_csv("hatecot_final_D3_style_1.csv")
df_hatecot_style_2 = pd.read_csv("hatecot_final_D3_style_2.csv")
df_hatecot_style_3 = pd.read_csv("hatecot_final_D3_style_3.csv")
df_hatecot_style_4 = pd.read_csv("hatecot_final_D3_style_4.csv")
df_hatecot_style_5 = pd.read_csv("hatecot_final_D3_style_5.csv")

In [72]:
df_hatecot_style = pd.concat([df_hatecot_style_1, df_hatecot_style_2, df_hatecot_style_3, df_hatecot_style_4, df_hatecot_style_5], ignore_index=True)
df_hatecot_style = df_hatecot_style.drop(columns=["style explanation"])

In [73]:
df_hatecot_style = df_hatecot_style[df_hatecot_style['domain'] != 'dynahate'].copy() # dynhate already in initial dataset

In [74]:
hate_labels = {
    "Hate",
    "Hate Speech",
    "Hateful",
    "Identity Directed Abuse",
    "Affiliation Directed Abuse",
    "Person Directed Abuse",
    "Toxic",
}

non_hate_labels = {
    "Neutral",
    "Benign",
    "Normal",
    "Not Hate Speech",
    "Not Offensive",
    "Not Hate",
    "Offensive",
}

In [75]:
def map_hateful_label(label):
    if label in hate_labels:
        return 1
    if label in non_hate_labels:
        return 0
    return None  # unclear / not necessarily hate

df_hatecot_style["hate_label"] = df_hatecot_style["label"].apply(map_hateful_label)
df_hatecot_style = df_hatecot_style[df_hatecot_style["hate_label"].notna()].copy()

df_hatecot_style['text'] = df_hatecot_style['post'].astype(str)
df_hatecot_style['target_conf'] = 1.0

In [76]:
df_hatecot_style.target.unique()
#pd.DataFrame(df_hatecot_style.target.unique()).to_csv(
 #   "hatecot_targets.csv",
  #  index=False
#)

array(['middle eastern', nan, 'muslim', 'Men', 'jewish',
       'Asian , Black or African American , Latino or non-white Hispanic , Middle Eastern , Native American or Alaska Native , Pacific Islander , Non-Hispanic White , Christian , Jewish , Mormon , Muslim , Men , Women',
       'people with physical disability',
       'Asian , Black or African American , Latino or non-white Hispanic , Middle Eastern , Native American or Alaska Native , Pacific Islander , Non-Hispanic White , Atheist , Buddhist , Christian , Hindu , Jewish , Mormon , Muslim , Immigrant , Non-binary people , Transgender men , Transgender (unspecififed) , Transgender women',
       'red haired people  |  Red-haired persons', 'Non-Hispanic White',
       'Christian , Atheist , Buddhist , Hindu , Jewish , Mormon , Muslim',
       'Women', 'lgbtq', 'Christian , Jewish , Muslim', 'women',
       'chinese',
       'People of other origin , People from some specific country',
       'native american', 'Muslim , People of 

In [77]:
hatecot_target_mapped = pd.read_csv("hatecot_targets_mapped.csv")

hatecot_target_mapped["category"] = (
    hatecot_target_mapped["category"]
    .astype(str)
    .str.strip()
    .str.lower()
)

In [78]:
import re
import pandas as pd

CANONICAL_TARGETS = {
    "none", "race", "religion", "gender", "sexual_orientation",
    "immigration_status", "nationality", "disability", "class",
    "politics", "violence", "health", "other"
}

def normalize_single_target(x):
    if pd.isna(x):
        return "none"

    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)

    if x in {"", "nan", "none", "unknown", "not specified"}:
        return "none"

    if any(k in x for k in [
        "asian", "black", "african american", "latino", "hispanic",
        "middle eastern", "native american", "alaska native",
        "pacific islander", "non-hispanic white", "white", "people of color",
        "poc", "non-white", "non white", "minorities", "minority",
        "mexican", "chinese", "indian", "arab", "arabic", "ethiopian",
        "racial", "ethnic", "race", "other races"
    ]):
        return "race"

    if any(k in x for k in [
        "muslim", "islam", "islamic", "christian", "catholic",
        "jewish", "jew", "mormon", "hindu", "buddhist", "atheist",
        "religion", "religious", "other relegions", "other religions",
        "holocaust"
    ]):
        return "religion"
    
    if any(k in x for k in [
        "lgbt", "lgbtq", "lgbtqa", "gay", "lesbian", "bisexual",
        "transgender", "trans women", "trans men", "sexual orientation",
        "sexual and gender minorities", "heterosexual", "incel",
        "prostitutes", "sex workers"
    ]):
        return "sexual_orientation"

    if any(k in x for k in [
        "men", "man", "women", "woman", "female", "male",
        "non-binary", "nonbinary", "other gender", "gender",
        "cis", "feminist", "feminists", "girls", "boys"
    ]):
        return "gender"

    if any(k in x for k in [
        "immigrant", "undocumented", "migrant worker", "migrant",
        "refugee", "foreigners"
    ]):
        return "immigration_status"

    if any(k in x for k in [
        "people from some specific country", "country", "nation",
        "american", "ethiopians", "ethiopia", "syrians", "syria",
        "pakistan", "india", "mexico", "japan", "chinese people",
        "russians", "russia", "germans", "irish", "scottish",
        "texas", "floridians", "samoa", "lebanese"
    ]):
        return "nationality"

    if any(k in x for k in [
        "physical disability", "disability", "disabled",
        "unspecified disability", "other disabilities",
        "cognitive disorders", "learning disabilities",
        "hearing impaired", "visually impaired", "autism",
        "aspergers", "dwarfism", "disabled folks"
    ]):
        return "disability"

    if any(k in x for k in [
        "poor", "rich", "working class", "class", "well off",
        "homeless"
    ]):
        return "class"

    if any(k in x for k in [
        "liberal", "conservative", "republican", "democrat",
        "right-wing", "left-wing", "alt-right", "communist",
        "socialist", "antifa", "fascist", "trump", "politic",
        "activists", "moderators", "capitalists", "nationalists",
        "pro choice", "abortion rights"
    ]):
        return "politics"

    if any(k in x for k in [
        "victim", "victims", "assault", "rape", "murder",
        "terrorism", "mass shooting", "war", "holocaust victims",
        "domestic abuse", "domestic violence", "molestation",
        "car accident", "violence", "genocide", "bombing",
        "child rape", "child sexual", "suicide victims",
        "police shooting"
    ]):
        return "violence"

    if any(k in x for k in [
        "mental health", "mental disorder", "mental illness",
        "physical illness", "health", "anorexic", "diabetics",
        "drug problems", "autism", "aspergers", "cancer",
        "seniors", "old folks", "elderly"
    ]):
        return "health"

    return "other"


def normalize_targets(x):
    if pd.isna(x):
        return ["none"]

    x = str(x).strip().lower()

    parts = [
        p.strip()
        for p in re.split(r"\s*\|\s*|\s*,\s*|/|;", x)
        if p.strip()
    ]

    mapped = [normalize_single_target(p) for p in parts]

    mapped = [m for m in mapped if m != "none"]

    mapped = list(dict.fromkeys(mapped))

    return mapped if mapped else ["none"]

In [79]:
target_map = dict(
    zip(
        hatecot_target_mapped["target"],
        hatecot_target_mapped["category"]
    )
)

def normalize_target(x):
    if pd.isna(x):
        return "none"

    return target_map.get(x, "other")

In [80]:
df_hatecot_style["target"] = df_hatecot_style["target"].apply(normalize_target)

In [81]:
from collections import Counter
import random

rng = random.Random(42)

def majority_target(series):
    all_targets = []

    for targets in series.dropna():
        if isinstance(targets, list):
            all_targets.extend(targets)
        else:
            all_targets.append(targets)

    if not all_targets:
        return "none"

    counts = Counter(all_targets)
    max_count = max(counts.values())
    top_targets = [k for k, v in counts.items() if v == max_count]

    return rng.choice(top_targets)

def majority_label(series):
    values = series.dropna().tolist()
    if len(values) == 0:
        return np.nan
    return Counter(values).most_common(1)[0][0]

In [82]:
df_hatecot_agg = (
    df_hatecot_style
    .groupby("text", as_index=False)
    .agg(
        avg=("hate_label", "mean"),
        hate_label=("hate_label", lambda x: int(x.mean() >= 0.5)),
        style=("style", majority_label),
        target=("target", majority_label),
        target_conf=("target_conf", "mean"),
    )
)

In [85]:
df_hatecot_agg.head(10)

,text,avg,hate_label,style,target,target_conf
0,"\r\n""But these laws, which allowed the Church, or a Christian, to appropriate the property of a Jew, can in no sense be considered as",1.0,1,0,religion,1.0
1,"\r\n- [black people are] lazy. they bring crime, they dont contribute positively. they are so lazy they dont even go to school.\r\n",1.0,1,1,nationality,1.0
2,\r\ncrayon:violet>tHIS pAiNTiNG iS bORiNG. iT iS ONLY eL,1.0,1,0,race,1.0
3,""" the core of the religion, alhamdulillah"" -Syed Ahmed ""I look at people on the street, and",0.0,0,1,religion,1.0
4,"""...the myth that suicides are trying to get attention by killing themselves.",1.0,1,0,health,1.0
5,"""I always find that trans activists want to take credit for all the work we do for gay rights, as evidenced in this particular case.""\n",1.0,1,0,sexual_orientation,1.0
6,"""IMPEACH TRUMP"" ""FREE PALESTINE"" ""DIRECT ACTION TO STOP GENOCIDE"" ""I",0.0,0,1,race,1.0
7,"""Islam is the most violent religion""\r\n- ""Malays are the same as the Indonesian/Singaporean""\r\n- ""Australia was discovered",1.0,1,0,race,1.0
8,"""Three [students] followed the school mule on a leash, behind them walked the son of the sexton, doffing his cap",0.0,0,1,health,1.0
9,"""You are ascribing homophobic thoughts that aren't in the posts. The only direct reference to homosexuality was that the sale was anti-gay, which",0.0,0,1,race,1.0


In [87]:
df_hatecot_agg.to_csv("hatecot_combined.csv", index=False)
df_hatecot_explicit = df_hatecot_agg[df_hatecot_agg["style"] == 1].copy()
df_hatecot_implicit = df_hatecot_agg[df_hatecot_agg["style"] == 0].copy()
df_hatecot_explicit.to_csv("hatecot_explicit.csv", index=False)
df_hatecot_implicit.to_csv("hatecot_implicit.csv", index=False)

In [88]:
df_hatecot_implicit.head(10)

,text,avg,hate_label,style,target,target_conf
0,"\r\n""But these laws, which allowed the Church, or a Christian, to appropriate the property of a Jew, can in no sense be considered as",1.0,1,0,religion,1.0
2,\r\ncrayon:violet>tHIS pAiNTiNG iS bORiNG. iT iS ONLY eL,1.0,1,0,race,1.0
4,"""...the myth that suicides are trying to get attention by killing themselves.",1.0,1,0,health,1.0
5,"""I always find that trans activists want to take credit for all the work we do for gay rights, as evidenced in this particular case.""\n",1.0,1,0,sexual_orientation,1.0
7,"""Islam is the most violent religion""\r\n- ""Malays are the same as the Indonesian/Singaporean""\r\n- ""Australia was discovered",1.0,1,0,race,1.0
11,"""dont let your girls grow up to be women, women include lesbians, divorces, single mothers, feminists, and gender equality activists"" a",1.0,1,0,gender,1.0
15,"""otherkin"" the only thing a dogkin hates more than people who keep their dog shedding around their house is people who have allergies",1.0,1,0,race,1.0
16,"""paranoid ideation"" : belief that unwelcome developments are being supervised or coerced by a omnipotent agency. ""Magical thinking"" < the",1.0,1,0,health,1.0
17,"""we can not ignore these countries because they're of strategic importance,"" he said during a speech at the washington institute for near east policy , a",1.0,1,0,race,1.0
26,"( female cats are ""crazy"" ) so this one is on the cat-lady, who probably having a cat and use that as a",1.0,1,0,gender,1.0


## Train, val, test for fine-tuning

In [2]:
from sklearn.model_selection import train_test_split
import pandas as pd

hatecot_combined = pd.read_csv("hatecot_combined.csv")

# 70% train, 30% temp
hatecot_train, hatecot_temp = train_test_split(
    hatecot_combined,
    test_size=0.30,
    random_state=42,
    stratify=hatecot_combined["hate_label"]
)

# 15% val, 15% test
hatecot_val, hatecot_test = train_test_split(
    hatecot_temp,
    test_size=0.50,
    random_state=42,
    stratify=hatecot_temp["hate_label"]
)

hatecot_train.to_csv("hatecot_train.csv", index=False)
hatecot_val.to_csv("hatecot_val.csv", index=False)
hatecot_test.to_csv("hatecot_test.csv", index=False)

print(len(hatecot_train), len(hatecot_val), len(hatecot_test))
print(hatecot_train["hate_label"].mean())
print(hatecot_val["hate_label"].mean())
print(hatecot_test["hate_label"].mean())

10370 2222 2223
0.4085824493731919
0.40864086408640865
0.40845704003598743
